# UEMR chốt, file 01 §2–4: vector thầy Omni `best` cho UniAV

Theo `docs/uemr_chot/UEMR_CHOT_01_UNIAV_BEST.md`. Encode mọi đoạn GT của 5.826 video train UniAV và 510 video kho bằng Omni `best`
(đường encode của cột (b)), ghi đúng định dạng thầy mà `libs/datasets/htstep.py` (HanLi `77fc4fa`) đọc. **Không train gì ở đây.**

- Nguồn của một đoạn, theo thứ tự: vector đã có trong cache lượt dev 1 (val) → clip cắt sẵn `GraceFeel/htstep-omni-clips` (cửa sổ trùng
  đoạn, lệch ≤ 0.01 s) → mp4 + `timestamps` → thiếu.
- Clip cắt sẵn đọc bằng loader của Omni-fix `ab772eb` (nhánh `htstep_lora`: bản vendored + nhánh `clip`), qua `OMNI_QWENVL_DIR`. Ô 6 kiểm
  đường này với `val_embeds_best.npz` (cos ≥ 0.999) trước khi encode cả tập.
- Colab Secrets: `HF_TOKEN` (đọc `GraceFeel/htstep-omni-clips`, ghi `HF_OUT`). GPU: A100. Đĩa `/content` trống ≥ 70 GB.
- **Chạy thử trước:** `SMOKE = True` (40 video train, ghi `smoke/`, không upload). Xong đổi `SMOKE = False`, Run all.

In [ ]:
# ===== 0. Cấu hình =====
FUSHION_REF = 'cdb2fac9b5b015637ef7c9fecd702e344f97a724'   # commit trên nhánh htstep (user push trước khi chạy)
SMOKE       = True       # True: 40 video train + mẫu kiểm, ghi smoke/, không upload. Xong đổi False rồi Run all
MISSING_MAX_PCT = 5.0    # plan 01 §2: tỉ lệ đoạn train thiếu vector thầy; chỉ user đổi (ghi lý do vào Nhật ký file 01)
HF_OUT      = 'GraceFeel/htstep-omni-av-best'   # dataset HF private nhận bộ thầy mới; '' = chỉ ghi Drive
HF_CLIPS    = 'GraceFeel/htstep-omni-clips'
RUN_ENCODE  = True       # ô 6-7 (GPU)
SEED        = 0
SEC_REF     = 0.32       # s/clip Omni best trên A100 (lượt dev 1: 0.314)
HANLI_REF, OMNI_FIX_REF = '77fc4fa', 'ab772eb'
PROTO1_SHA16 = 'a4ca43db9829842a'   # protocol lượt dev 1: nguồn hash của annotations, manifest, adapter và cache vector val
FLAGS = {k: v for k, v in dict(globals()).items() if k in ('FUSHION_REF', 'SMOKE', 'MISSING_MAX_PCT', 'HF_OUT', 'RUN_ENCODE', 'SEED')}

In [ ]:
# ===== 1. Drive, code (Fushion, HanLi, Omni-fix), thư viện, unit test =====
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, sys, json, glob, shutil, subprocess, time, hashlib, platform, csv, tarfile
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'
os.environ.pop('HF_XET_HIGH_PERFORMANCE', None)
T_START = time.time()
FUSHION, HANLI, OMNI_FIX = '/content/code/Fushion_UEMR', '/content/code/HanLi-Retriever', '/content/code/Omni-fix'
os.makedirs('/content/code', exist_ok=True)

def clone(url, path, ref, *branch):
    os.chdir('/content')
    shutil.rmtree(path, ignore_errors=True)
    subprocess.run(['git', 'clone', '-q', *branch, url, path], check=True)
    r = subprocess.run(['git', '-C', path, 'checkout', '-q', ref], capture_output=True, text=True)
    assert r.returncode == 0, f'không checkout được {ref} của {url}: {r.stderr} (đã push chưa?)'
    return subprocess.run(['git', '-C', path, 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip()

COMMIT = clone('https://github.com/chien24/Fushion_UEMR.git', FUSHION, FUSHION_REF)
HANLI_COMMIT = clone('https://github.com/im-xiaoming/HanLi-Retriever.git', HANLI, HANLI_REF)
OMNI_COMMIT = clone('https://github.com/im-xiaoming/Omni-Retriever-Custom.git', OMNI_FIX, OMNI_FIX_REF, '-b', 'htstep_lora')
print('Fushion :', COMMIT); print('HanLi   :', HANLI_COMMIT); print('Omni-fix:', OMNI_COMMIT)
if FUSHION not in sys.path:
    sys.path.insert(0, FUSHION)
os.environ['PYTHONPATH'] = FUSHION + ':' + os.environ.get('PYTHONPATH', '')
os.chdir(FUSHION)

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr)[-3000:]
    if out.strip():
        print(out)
    if check and r.returncode:
        raise RuntimeError(f'lệnh lỗi ({r.returncode}): {cmd[:200]}')
    return r

def check(name, ok, detail=''):
    print(f"CHECK {name}: {'OK' if ok else 'FAIL'}" + (f' | {detail}' if detail else ''))
    assert ok, f'CHECK {name} FAIL: {detail}'

sh(f'pip install -q -r {FUSHION}/requirements.txt')
sh('pip uninstall -y -q torchao', check=False)
import numpy as np, pandas as pd, torch
from IPython.display import display
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có'
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0.0
ENV = {'python': platform.python_version(), 'torch': torch.__version__, 'numpy': np.__version__, 'gpu': gpu, 'gpu_GB': round(GPU_GB, 1),
       'fushion': COMMIT, 'hanli': HANLI_COMMIT, 'omni_fix': OMNI_COMMIT, 'smoke': SMOKE}
print(json.dumps(ENV, indent=1, ensure_ascii=False))
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_teacher_best.py', 'tests/test_uemr_htstep.py', '-q', '-p', 'no:cacheprovider'], cwd=FUSHION,
                   capture_output=True, text=True, env={**os.environ, 'UNIAV_REPO': HANLI})
print(r.stdout[-1200:])
check('unit test (gồm loader thầy của HanLi)', r.returncode == 0 and ' skipped' not in r.stdout.splitlines()[-1], r.stdout.splitlines()[-1] if r.stdout else r.stderr[-300:])

def sha256(path, bs=1 << 24):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(bs), b''):
            h.update(chunk)
    return h.hexdigest()

def wj(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=1, ensure_ascii=False, default=str)
    os.replace(tmp, path)

def rj(path, default=None):
    return json.load(open(path, encoding='utf-8')) if os.path.exists(path) else default

DRIVE = '/content/drive/MyDrive'
KL    = f'{DRIVE}/Colab Notebooks/code KL/Fushion'
H     = f'{KL}/uemr_chot_run/teacher_best' + ('/smoke' if SMOKE else '')      # mọi đầu ra của notebook này
DB1   = f'{KL}/uemr_htstep_run/omni/best/db'                                  # cache vector Omni best của lượt dev 1 (chỉ đọc)
RUN_DRIVE = f'{DRIVE}/omniretriever/htstep_lora/lora_bs32'                    # lượt train Omni (adapter best, val_embeds_best.npz)
LOCAL = '/content/smoke' if SMOKE else '/content/full'
A, CLIPS, VID, FEAT, OUT = '/content/hts', '/content/htstep_clips', '/content/videos_raw', '/content/k400_sample', f'{LOCAL}/omni_av_best'
if SMOKE:
    shutil.rmtree(H, ignore_errors=True); shutil.rmtree(LOCAL, ignore_errors=True)
for d in (H, A, CLIPS, VID, FEAT, LOCAL, '/content/videos'):
    os.makedirs(d, exist_ok=True)
INV = rj(f'{H}/inventory.json', {})
def note(**kw):
    INV.update(kw); wj(f'{H}/inventory.json', INV)

In [ ]:
# ===== 2. Adapter best; hash theo protocol lượt dev 1 =====
PR = json.load(open(f'{FUSHION}/configs/htstep_uemr_protocol.json', encoding='utf-8'))
check('protocol lượt dev 1 (nguồn hash)', sha256(f'{FUSHION}/configs/htstep_uemr_protocol.json')[:16] == PROTO1_SHA16)
D, M = PR['data'], PR['models']
BEST_ADAPTER = '/content/adapters/best'
if not os.path.exists(f'{BEST_ADAPTER}/adapter_model.safetensors'):
    src = f'{RUN_DRIVE}/eval/adapter_best'
    if os.path.exists(f'{src}/adapter_model.safetensors'):
        shutil.copytree(src, BEST_ADAPTER, dirs_exist_ok=True)
    else:
        sh("hf download GraceFeel/omni_lora_htstep --include 'lora_bs32/adapter_best/*' --local-dir /content/adapters/_hf")
        shutil.copytree('/content/adapters/_hf/lora_bs32/adapter_best', BEST_ADAPTER, dirs_exist_ok=True)
BEST_SHA = sha256(f'{BEST_ADAPTER}/adapter_model.safetensors')
check('sha256 adapter best = 5e9899aa…', BEST_SHA == M['omni_best']['sha256'], BEST_SHA[:16])

In [ ]:
# ===== 3. Danh sách đoạn cần vector thầy và nguồn của từng đoạn (plan §2); CHECK tỉ lệ thiếu =====
from huggingface_hub import hf_hub_download, HfApi
from uemr_htstep import data as hd, teacher_best as tb
t0 = time.time()
fetch_meta = lambda name: hf_hub_download(HF_CLIPS, name, repo_type='dataset', local_dir=A)
val_p, train_p, index_p = fetch_meta(D['val_manifest']['file']), fetch_meta(D['train_manifest']['file']), fetch_meta('meta/precut_index.jsonl')
check('sha256 val manifest', sha256(val_p) == D['val_manifest']['sha256'])
check('sha256 train manifest', sha256(train_p) == D['train_manifest']['sha256'])
HTREPO = f'{A}/htstep_repo'
if not os.path.isdir(f'{HTREPO}/.git'):
    subprocess.run(['git', 'clone', '-q', D['annotations']['repo'], HTREPO], check=True)
subprocess.run(['git', '-C', HTREPO, 'checkout', '-q', D['annotations']['commit']], check=True)
ANN_P, TAX_P = f'{HTREPO}/{D["annotations"]["file"]}', f'{HTREPO}/{D["taxonomy"]["file"]}'
check('sha256 annotations.json chính thức', sha256(ANN_P) == D['annotations']['sha256'])
check('sha256 taxonomy.csv chính thức', sha256(TAX_P) == D['taxonomy']['sha256'])
check('sha256 configs/htstep_train_ids.json', sha256(f"{FUSHION}/{D['train_ids']['file']}") == D['train_ids']['sha256'])
check('sha256 tar_index.csv', sha256(f'{OMNI_FIX}/data/htstep/tar_index.csv') == D['tar_index']['sha256'])
ANN = json.load(open(ANN_P, encoding='utf-8'))
IDS = json.load(open(f"{FUSHION}/{D['train_ids']['file']}", encoding='utf-8'))
KHO = D['gallery']
TRAIN_ROWS, VAL_ROWS = hd.read_jsonl(train_p), hd.read_jsonl(val_p)
TAR = {r['video_id']: (r['tar_path'], int(r['offset_data']), int(r['size'])) for r in csv.DictReader(open(f'{OMNI_FIX}/data/htstep/tar_index.csv', newline='', encoding='utf-8'))}
INDEX = {r['id']: r for r in hd.read_jsonl(index_p)}

# cache lượt dev 1: vector Omni best của mọi đoạn GT val (dòng R7), khoá video__ts_te
if not os.path.exists(f'{DB1}/segments.npz'):
    DB1 = os.path.dirname(hf_hub_download('GraceFeel/encode_omni', f'htstep_uemr/{PROTO1_SHA16[:8]}/best/db/segments.npz', repo_type='dataset', local_dir=A))
    hf_hub_download('GraceFeel/encode_omni', f'htstep_uemr/{PROTO1_SHA16[:8]}/best/db/meta.json', repo_type='dataset', local_dir=A)
META1 = rj(f'{DB1}/meta.json')
check('cache dev 1: protocol a4ca43db, adapter best', META1['protocol_sha16'] == PROTO1_SHA16 and META1['adapter_sha256'] == BEST_SHA)
check('cache dev 1: sha256 segments.npz = meta.json', sha256(f'{DB1}/segments.npz') == META1['sha256']['segments.npz'])
with np.load(f'{DB1}/segments.npz') as z:
    _k, _first = np.unique(z['seg_key'].astype(str), return_index=True)
    CACHE = dict(zip(_k.tolist(), z['emb'][_first]))
print('cache dev 1:', len(CACHE), 'clip khác nhau')

SEG = {'train': tb.assign(tb.gt_segments(ANN, IDS['uniav_train']), TRAIN_ROWS, set(), set(TAR)),
       'val': tb.assign(tb.gt_segments(ANN, KHO), VAL_ROWS, set(CACHE), set())}
def count(segs):
    return dict(Counter(s['source'] + (f": {s['why']}" if 'why' in s else '') for s in segs))
CNT = {k: {'video': len({s['video_id'] for s in v}), 'đoạn': len(v), **count(v)} for k, v in SEG.items()}
display(pd.DataFrame(CNT).T.fillna(0).astype(int))
# số đếm ở local ngày 2026-11-12 (Nhật ký file 01): danh sách trên Colab phải giống hệt
check('số đoạn train = Nhật ký', (CNT['train']['đoạn'], CNT['train'].get('clip'), CNT['train'].get('mp4'), CNT['train'].get('missing: no mp4')) == (36026, 33326, 379, 2321), str(CNT['train']))
check('mọi đoạn GT val (2.990, 510 video) đã có trong cache dev 1', CNT['val'] == {'video': 510, 'đoạn': 2990, 'cache': 2990}, str(CNT['val']))
no_mp4 = sorted({s['video_id'] for s in SEG['train'] if s['source'] == 'missing'})
LOW = 100 * sum(s['source'] == 'missing' for s in SEG['train']) / len(SEG['train'])
print(f'đoạn train chắc chắn thiếu (video không có mp4 trong nguyenminh04/htstep-videos): {LOW:.2f} % = {len(no_mp4)} video.')
print('Thầy cũ (nguyenminh04/htstep-omni-av) cũng không có đúng các video này: 5.465 / 5.826 video train có file thầy (Nhật ký file 01).')
note(counts=CNT, missing_lower_bound_pct=LOW, videos_no_mp4=no_mp4)
if SMOKE and LOW > MISSING_MAX_PCT:      # SMOKE chỉ thử đường chạy trên 40 video, không ghi bộ thầy thật
    print(f'[smoke] LƯU Ý: lượt thật sẽ dừng ở đây vì {LOW:.2f} % > {MISSING_MAX_PCT} % (plan 01 §2), cho tới khi user quyết định')
else:
    check(f'tỉ lệ đoạn train thiếu <= {MISSING_MAX_PCT} % (cận dưới, trước khi encode)', LOW <= MISSING_MAX_PCT, f'{LOW:.2f} %: dừng, báo user (plan 01 §2)')

In [ ]:
# ===== 4. Một tar đặc trưng K400 của mỗi tập (cho CHECK đọc lại bằng htstep.py ở ô 9); tập con SMOKE =====
api = HfApi()
def feat_sample(tar):      # tar ~0.1 GB; 1000 = 90 video kho, 1006 = 71 video train (đếm từ header tar, 2026-11-09)
    p = hf_hub_download('nguyenminh04/htstep-feats-k400', tar, repo_type='dataset', local_dir=f'{A}/feat_tars')
    with tarfile.open(p) as tf:
        names = [m for m in tf if m.isfile() and m.name.endswith('.npz') and m.size > 0]
        for m in names:
            with tf.extractfile(m) as src, open(f'{FEAT}/{os.path.basename(m.name)}', 'wb') as dst:
                shutil.copyfileobj(src, dst)
    os.remove(p)
    return tar, len(names)
if not os.path.exists(f'{FEAT}/.done'):
    print([feat_sample(t) for t in ('feats/htstep_feats_1000.tar', 'feats/htstep_feats_1006.tar')])
    open(f'{FEAT}/.done', 'w').close()
FEAT_VIDS = {f[:-4] for f in os.listdir(FEAT) if f.endswith('.npz')}
with_clip = sorted({s['video_id'] for s in SEG['train'] if s['source'] == 'clip'} & FEAT_VIDS)
check('tar mẫu đủ một batch (32) video train có clip và video kho', len(with_clip) >= 32 and len(FEAT_VIDS & set(KHO)) >= 32, f'{len(with_clip)} train, {len(FEAT_VIDS & set(KHO))} kho')
if SMOKE:
    keep = set(with_clip[:40]) | set(sorted({s['video_id'] for s in SEG['train'] if s['source'] == 'mp4'})[:3])
    SEG['train'] = [s for s in SEG['train'] if s['video_id'] in keep]
    print(f"[smoke] train: {len(keep)} video, {len(SEG['train'])} đoạn {count(SEG['train'])}")

In [ ]:
# ===== 5. Clip cắt sẵn (theo shard, resume, kiểm sha256 từng clip), mp4 cho các đoạn không có clip =====
import importlib.util as _ilu
_spec = _ilu.spec_from_file_location('htstep_io', f'{OMNI_FIX}/src/omniretriever/data/htstep_io.py')
hio = _ilu.module_from_spec(_spec); _spec.loader.exec_module(hio)
t0 = time.time()
rng = np.random.default_rng(SEED)
BY_VAL = {r['id']: r for r in VAL_ROWS}
CHK_VAL = sorted(rng.choice(sorted(BY_VAL), 20, replace=False).tolist())                       # 20 clip GT val: so với val_embeds_best.npz
_tc = [s for s in SEG['train'] if s['source'] == 'clip']
CHK_TRAIN = [_tc[i] for i in sorted(rng.choice(len(_tc), 20, replace=False).tolist())]         # 20 clip train: encode hai lần
clip_id = lambda path: path[len('clips/'):-len('.npz')].replace('/', '#')                      # clips/<video>/<k>.npz -> id của precut_index
need = {clip_id(s['clip']) for s in SEG['train'] if s['source'] == 'clip'} | set(CHK_VAL)
missing_idx = [i for i in need if i not in INDEX or INDEX[i]['status'] != 'ok']
check('mọi clip cần dùng có trong precut_index (status ok)', not missing_idx, str(missing_idx[:5]))
shards = sorted({INDEX[i]['shard'] for i in need})
sizes = {f.path: f.size for f in api.list_repo_tree(HF_CLIPS, repo_type='dataset', path_in_repo='shards') if f.path.endswith('.tar')}
check('tên shard trong precut_index khớp repo', all(f'shards/{s}.tar' in sizes for s in shards), str([s for s in shards if f'shards/{s}.tar' not in sizes][:3]))
done_p = f'{CLIPS}/fetched.json'
FETCHED = rj(done_p, {})
todo = [s for s in shards if f'shards/{s}.tar' not in FETCHED]
need_gb = (sum(sizes[f'shards/{s}.tar'] for s in todo) + max([sizes[f'shards/{s}.tar'] for s in todo], default=0)) / 2**30 + 15
free = shutil.disk_usage('/content').free / 2**30
print(f'shard cần: {len(shards)} / {len(sizes)}, còn tải {len(todo)} (cần {need_gb:.0f} GB, trống {free:.0f} GB)')
check('đĩa /content đủ cho clip', free >= need_gb, f'{free:.0f} GB < {need_gb:.0f} GB')
for i, s in enumerate(todo, 1):
    tar = hf_hub_download(HF_CLIPS, f'shards/{s}.tar', repo_type='dataset', local_dir=f'{CLIPS}/_dl')
    with tarfile.open(tar) as tf:
        tf.extractall(CLIPS, filter='data')
    os.remove(tar)
    FETCHED[f'shards/{s}.tar'] = True
    wj(done_p, FETCHED)
    if i % 5 == 0 or i == len(todo):
        print(f'[clip] {i}/{len(todo)} shard, {(time.time() - t0) / 60:.1f} phút', flush=True)
def clip_ok(i):
    p = f"{CLIPS}/{INDEX[i]['clip']}"
    return os.path.exists(p) and sha256(p) == INDEX[i]['sha256']
with ThreadPoolExecutor(8) as ex:
    bad = [i for i, ok in zip(sorted(need), ex.map(clip_ok, sorted(need))) if not ok]
check(f'sha256 của {len(need)} clip = precut_index', not bad, str(bad[:5]))

# mp4: đoạn nguồn 'mp4' (video có trong tar nhưng clip không cắt được / cửa sổ lệch) + 10 clip train của phép so mp4 / clip
CHK_MP4 = CHK_TRAIN[:10]
mp4_vids = sorted({s['video_id'] for s in SEG['train'] if s['source'] == 'mp4'} | {s['video_id'] for s in CHK_MP4})
def fetch_one(v):
    dst = f'/content/videos/{v}.mp4'           # bản decord đọc được (AV1 đã chuyển H.264 không mất mát, như precut)
    if os.path.exists(dst) and os.path.getsize(dst):
        return v, 'có sẵn'
    raw = f'{VID}/{v}.mp4'
    try:
        tar, off, size = TAR[v]
        hio.range_get(tar, off, size, raw)
        p = hio.probe(raw)
        if p['codec'] != 'av1':
            shutil.move(raw, dst)
            return v, 'tải'
        ok, why, _ = hio.transcode(raw, dst + '.h264.mp4', p)
        os.remove(raw)
        if not ok:
            return v, f'lỗi transcode: {why}'
        os.replace(dst + '.h264.mp4', dst)
        return v, 'av1 -> h264'
    except Exception as e:     # noqa: BLE001 - một video lỗi không làm mất các video khác
        return v, f'lỗi: {type(e).__name__}: {e}'
with ThreadPoolExecutor(4) as ex:
    MP4 = dict(ex.map(fetch_one, mp4_vids))
print('mp4:', dict(Counter(s.split(':')[0] for s in MP4.values())))
MP4_BAD = {v: s for v, s in MP4.items() if s.startswith('lỗi')}
note(mp4_status=dict(Counter(s.split(':')[0] for s in MP4.values())), mp4_failed=MP4_BAD, shards=len(shards), sec_fetch=round(time.time() - t0, 1))

In [ ]:
# ===== 6. CHECK đường encode trước khi chạy cả tập (plan §3) =====
from omni_retrieval.manifest import PROMPT, store_keys, read_store, write_jsonl, seg_key
BASE_MODEL, BEATS = '/content/WAVE_HOME/WAVE-7B', '/content/WAVE_HOME/BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2.pt'

def wave_ok():
    idx = glob.glob(f'{BASE_MODEL}/*.index.json')
    return os.path.exists(BEATS) and bool(idx) and all(os.path.exists(f'{BASE_MODEL}/{s}') for s in set(json.load(open(idx[0]))['weight_map'].values()))

def omni_env():
    check('GPU đủ cho Omni 7B (>= 22 GB)', GPU_GB >= 22, f'{gpu}, {GPU_GB:.0f} GB')
    chk = subprocess.run([sys.executable, '-c', 'import decord, av, peft, torch, transformers; print(transformers.__version__, torch.cuda.is_available())'], capture_output=True, text=True)
    last = (chk.stdout.strip().splitlines() or [''])[-1]
    check('môi trường encode (transformers 4.51.3, CUDA)', chk.returncode == 0 and last.split()[:2] == ['4.51.3', 'True'], f'{last} {chk.stderr[-300:]}')
    if not wave_ok():
        sh('hf download nguyenminh04/omni-model --local-dir /content/WAVE_HOME --max-workers 2')
    check('WAVE-7B + BEATs đủ file', wave_ok())
    # loader của Omni-fix ab772eb: bản vendored của Fushion + record 'clip' (clip cắt sẵn); record video không đổi
    return (f'cd {FUSHION} && OMNI_QWENVL_DIR={OMNI_FIX}/training CLIP_ROOT={CLIPS} python -m omni_retrieval.encode --base-model {BASE_MODEL} '
            f'--beats-path {BEATS} --adapter {BEST_ADAPTER} --video-root /content/videos')

def encode(manifest, store, log, enc, extra=''):
    # | tee che mã lỗi của tiến trình con nên tự đếm lại từ store
    with open(log, 'a') as f:
        f.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} =====\n")
    get_ipython().system(f'{enc} --modality av --manifest "{manifest}" --store "{store}" {extra} 2>&1 | tee -a "{log}"')
    todo = [json.loads(l)['id'] for l in open(manifest) if l.strip()]
    return sorted(set(todo) - store_keys(store, 'av'))

cosv = lambda a, b: float(np.asarray(a, np.float32) @ np.asarray(b, np.float32) / np.linalg.norm(np.asarray(a, np.float32)) / np.linalg.norm(np.asarray(b, np.float32)))
rec_clip = lambda rid, clip: {'id': rid, 'type': 'retrieval', 'conversations': [{'from': 'human', 'value': PROMPT}], 'clip': clip}
if RUN_ENCODE:
    t0 = time.time()
    ENC = omni_env()
    P = f'{LOCAL}/probe'
    shutil.rmtree(P, ignore_errors=True); os.makedirs(P)
    # (a) 20 clip GT val qua clip cắt sẵn, so với val_embeds_best.npz (vector mà eval của best đã chấm)
    EMB = f'{RUN_DRIVE}/eval/val_embeds_best.npz'
    check('có val_embeds_best.npz trên Drive', os.path.exists(EMB), EMB)
    with np.load(EMB) as z:
        row = {x: i for i, x in enumerate(z['ids'].astype(str))}
        REF = {r: z['av'][row[r]].astype(np.float32) for r in CHK_VAL}
    write_jsonl(f'{P}/val.jsonl', [rec_clip(r, BY_VAL[r]['clip']) for r in CHK_VAL])
    left = encode(f'{P}/val.jsonl', f'{P}/val', f'{H}/probe.log', ENC, '--num-workers 0')
    check('encode đủ 20 clip val', not left, str(left))
    va = read_store(f'{P}/val', 'av')
    ca = {r: cosv(va[r], REF[r]) for r in CHK_VAL}
    print('cos với val_embeds_best: mean %.5f, min %.5f' % (np.mean(list(ca.values())), min(ca.values())))
    check('clip cắt sẵn = val_embeds_best.npz (cos >= 0.999, trung bình và từng clip)', min(ca.values()) >= 0.999, str({k: round(v, 5) for k, v in ca.items() if v < 0.999}))
    # cùng 20 đoạn trong cache dev 1 (mp4 + timestamps làm tròn 0.01 s): chỉ báo, nối hai đường của val
    cc = {r: cosv(va[r], CACHE[seg_key(BY_VAL[r]['video_id'], round(BY_VAL[r]['window'][0], 2), round(BY_VAL[r]['window'][1], 2))]) for r in CHK_VAL}
    print('cos với cache dev 1 (đường mp4): mean %.5f, min %.5f (chỉ báo)' % (np.mean(list(cc.values())), min(cc.values())))
    # (b) 20 clip train encode hai lần
    write_jsonl(f'{P}/train.jsonl', [tb.record(s) for s in CHK_TRAIN])
    for k in ('t1', 't2'):
        left = encode(f'{P}/train.jsonl', f'{P}/{k}', f'{H}/probe.log', ENC, '--num-workers 0')
        check(f'encode đủ 20 clip train ({k})', not left, str(left))
    t1, t2 = read_store(f'{P}/t1', 'av'), read_store(f'{P}/t2', 'av')
    cr = {s['id']: cosv(t1[s['id']], t2[s['id']]) for s in CHK_TRAIN}
    print('cos hai lần encode: min %.6f' % min(cr.values()))
    check('20 clip train encode hai lần cho cos 1.0', round(min(cr.values()), 4) == 1.0, str({k: v for k, v in cr.items() if round(v, 4) < 1}))
    # (c) 10 đoạn: mp4 cắt bằng timestamps so với clip cắt sẵn (chỉ báo)
    ok10 = [s for s in CHK_MP4 if s['video_id'] not in MP4_BAD]
    write_jsonl(f'{P}/mp4.jsonl', [tb.record({**s, 'source': 'mp4'}) for s in ok10])
    left = encode(f'{P}/mp4.jsonl', f'{P}/mp4', f'{H}/probe.log', ENC, '--num-workers 0')
    m = read_store(f'{P}/mp4', 'av')
    cm = {s['id']: cosv(m[s['id']], t1[s['id']]) for s in ok10 if s['id'] in m}
    cm_stat = {'n': len(cm), 'mean': float(np.mean(list(cm.values()))) if cm else None, 'min': min(cm.values()) if cm else None, 'cos': cm, 'not_encoded': left}
    print('mp4 + timestamps so với clip cắt sẵn (chỉ báo):', {k: v for k, v in cm_stat.items() if k != 'cos'})
    spc = json.loads(open(f'{P}/t1/runs.jsonl').read().strip().splitlines()[-1]).get('sec_per_clip')
    note(encode_check={'val_vs_embeds_best': {'mean': float(np.mean(list(ca.values()))), 'min': min(ca.values()), 'ids': CHK_VAL},
                       'val_vs_dev1_cache': {'mean': float(np.mean(list(cc.values()))), 'min': min(cc.values())},
                       'train_twice_min': min(cr.values()), 'mp4_vs_clip': cm_stat,
                       'sec_per_clip': spc, 'sec': round(time.time() - t0, 1)})

In [ ]:
# ===== 7. Encode mọi đoạn train (resume theo store trên Drive) =====
STORE = f'{H}/store'
todo_segs = [s for s in SEG['train'] if s['source'] == 'clip' or (s['source'] == 'mp4' and s['video_id'] not in MP4_BAD)]
if RUN_ENCODE:
    t0 = time.time()
    have = store_keys(STORE, 'av')
    recs = [tb.record(s) for s in todo_segs if s['id'] not in have]
    print(f'đoạn cần encode: {len(recs)} / {len(todo_segs)} (clip {sum(s["source"] == "clip" for s in todo_segs)}, mp4 {sum(s["source"] == "mp4" for s in todo_segs)}) '
          f'| ước lượng {len(recs) * SEC_REF / 3600:.2f} giờ ở {SEC_REF} s/clip')
    if recs:
        write_jsonl(f'{H}/segs_todo.jsonl', recs)
        if len(recs) > 20:
            encode(f'{H}/segs_todo.jsonl', STORE, f'{H}/encode.log', ENC, '--num-workers 4 --limit 20')
            spc = json.loads(open(f'{STORE}/runs.jsonl').read().strip().splitlines()[-1]).get('sec_per_clip') or float('nan')
            print(f'20 clip: {spc:.3f} s/clip -> còn {len(recs) - 20} clip ≈ {(len(recs) - 20) * spc / 3600:.2f} giờ')
        encode(f'{H}/segs_todo.jsonl', STORE, f'{H}/encode.log', ENC, '--num-workers 4')
    note(sec_encode=round(time.time() - t0, 1) + INV.get('sec_encode', 0), encoded_now=len(recs))
have = store_keys(STORE, 'av')
err = {}
if os.path.exists(f'{STORE}/failed.jsonl'):
    for l in open(f'{STORE}/failed.jsonl'):
        if l.strip():
            q = json.loads(l); err[q['id']] = q['error']
FAIL = [{'id': s['id'], 'source': s['source'], 'error': err.get(s['id'], 'không có trong failed.jsonl')} for s in todo_segs if s['id'] not in have]
pd.DataFrame(FAIL, columns=['id', 'source', 'error']).to_csv(f'{H}/failed.csv', index=False)
print(f'encode lỗi: {len(FAIL)} đoạn {dict(Counter(f["source"] for f in FAIL))} -> {H}/failed.csv')

In [ ]:
# ===== 8. Ghi bộ thầy mới đúng định dạng HanLi (plan §4): train/<id>.npz, val/<id>.npz, config.json =====
t0 = time.time()
shutil.rmtree(OUT, ignore_errors=True)
vec = read_store(STORE, 'av')
W = {'train': tb.write(SEG['train'], vec, f'{OUT}/train'),
     'val': tb.write(SEG['val'], {s['id']: CACHE[s['key']] for s in SEG['val'] if s['source'] == 'cache'}, f'{OUT}/val')}
del vec
_miss = set(W['train']['missing'])
why = {s['id']: (s.get('why') or ('mp4 không tải / chuyển được' if s['video_id'] in MP4_BAD else 'encode lỗi')) for s in SEG['train'] if s['id'] in _miss}
PCT = {k: 100 * len(W[k]['missing']) / len(SEG[k]) for k in W}
print({k: {'file': len(W[k]['files']), 'đoạn có vector': W[k]['segments'], 'đoạn thiếu': len(W[k]['missing']), '% thiếu': round(PCT[k], 2)} for k in W})
print('lý do thiếu (train):', dict(Counter(why.values())))
CFG = {'teacher': 'Omni best = WAVE-7B (nguyenminh04/omni-model) + LoRA HT-Step GraceFeel/omni_lora_htstep lora_bs32/adapter_best',
       'adapter_sha256': BEST_SHA, 'embed_dim': 3584, 'dtype': 'float16', 'normalize': True,
       'encode': 'Fushion omni_retrieval.encode (cột (b)): LazySupervisedDataset, decord, 8 khung, 224 px (min = max_pixels 50176), tiếng 8 s, batch 1, bf16; '
                 'loader Omni-fix ab772eb qua OMNI_QWENVL_DIR',
       'sources': {'cache': 'val: vector R7 của lượt dev 1 (mp4, cửa sổ làm tròn 0.01 s), GraceFeel/encode_omni htstep_uemr/a4ca43db/best',
                   'clip': 'clip cắt sẵn GraceFeel/htstep-omni-clips, cửa sổ = đoạn annotation (lệch <= 0.01 s)',
                   'mp4': 'nguyenminh04/htstep-videos, timestamps = đoạn annotation (AV1 chuyển H.264 không mất mát)'},
       'format': 'như nguyenminh04/htstep-omni-av: <split>/<video_id>.npz, emb (k, 3584) fp16, seg_idx = chỉ số trong annotations, step_id, segment, partial',
       'window': 'đoạn annotation [start, end]; đoạn end <= start bỏ', 'fushion_commit': COMMIT, 'omni_fix_commit': OMNI_COMMIT, 'hanli_commit': HANLI_COMMIT,
       'annotations_sha256': D['annotations']['sha256'], 'smoke': SMOKE, 'created': time.strftime('%Y-%m-%d %H:%M:%S'),
       'counts': {k: {'videos': len({s['video_id'] for s in SEG[k]}), 'segments': len(SEG[k]), 'by_source': count(SEG[k]), 'files': len(W[k]['files']),
                      'segments_with_vector': W[k]['segments'], 'segments_missing': len(W[k]['missing']), 'missing_pct': PCT[k]} for k in W},
       'missing': {'train': why, 'val': W['val']['missing']}, 'encode_check': INV.get('encode_check'),
       'sha256': {f'{k}/{f}': h for k in W for f, h in W[k]['files'].items()}}
wj(f'{OUT}/config.json', CFG)
note(written={k: CFG['counts'][k] for k in W}, missing_reasons=dict(Counter(why.values())))
check('val: đủ vector cho mọi đoạn GT của 510 video', not W['val']['missing'] and len(W['val']['files']) == 510)
if not SMOKE:
    check(f'tỉ lệ đoạn train thiếu <= {MISSING_MAX_PCT} %', PCT['train'] <= MISSING_MAX_PCT, f"{PCT['train']:.2f} %")

In [ ]:
# ===== 9. CHECK: đọc lại bằng chính htstep.py của HanLi (một batch train, một batch val) =====
if HANLI not in sys.path:
    sys.path.insert(0, HANLI)
from libs.core import cfg as HCFG
from libs.datasets import make_dataset
step_emb = hf_hub_download('nguyenminh04/htstep-text-feats', 'text_feats/step_emb_iv2j.npz', repo_type='dataset', local_dir=A)
data_cfg = dict(HCFG['dataset'], json_file=ANN_P, taxonomy_file=TAX_P, step_emb_file=step_emb, feat_folder=FEAT, omni_emb_folder=OUT,
                scale_factor=HCFG['model']['scale_factor'], regression_range=HCFG['model']['regression_range'], backbone_arch=HCFG['model']['backbone_arch'])
READ = {}
for split, training, key in ((['train'], True, 'train'), (['val_seen'], False, 'val')):
    ds = make_dataset('htstep', training, split, **data_cfg)
    idx = [i for i, x in enumerate(ds.data_list) if f"{x['id']}.npz" in W[key]['files']][:HCFG['batch_size']]
    check(f'{key}: đủ một batch video có file thầy trong tar mẫu', len(idx) == HCFG['batch_size'], f'{len(idx)} / {len(ds)} video')
    n_ok = n_gt = 0
    worst, norm_lo, norm_hi = 0.0, 9.0, 0.0
    for i in idx:
        item, meta = ds[i], ds.data_list[i]
        seg, ok = item['omni_seg'].numpy(), item['omni_ok'].numpy()
        assert seg.shape == (HCFG['dataset']['num_classes'], 3584) and np.isfinite(seg).all(), (meta['id'], seg.shape)
        with np.load(f"{OUT}/{key}/{meta['id']}.npz") as z:
            mine = {int(a): e.astype(np.float32) for a, e in zip(z['seg_idx'], z['emb'])}
        for ann, slot in meta['ann_slots'].items():
            assert bool(ok[slot]) == (ann in mine), (meta['id'], ann)
            if ann in mine:
                worst = max(worst, float(np.abs(seg[slot] - mine[ann] / np.linalg.norm(mine[ann])).max()))
        n = np.linalg.norm(seg[ok], axis=1)
        norm_lo, norm_hi = min(norm_lo, float(n.min())), max(norm_hi, float(n.max()))
        n_ok += int(ok.sum()); n_gt += len(meta['ann_slots'])
    READ[key] = {'videos': len(idx), 'gt_segments': n_gt, 'with_teacher': n_ok, 'norm_min': norm_lo, 'norm_max': norm_hi, 'max_abs_diff': worst}
    print(key, READ[key])
    check(f'{key}: htstep.py đọc đúng (48 x 3584, không NaN, norm ≈ 1, trùng vector đã ghi)', n_ok > 0 and abs(norm_lo - 1) < 1e-3 and abs(norm_hi - 1) < 1e-3 and worst < 1e-3)
note(readback=READ)

In [ ]:
# ===== 10. Lưu Drive, upload HF, tóm tắt cho Nhật ký =====
TAR_OUT = f'{H}/htstep-omni-av-best.tar'
with tarfile.open(TAR_OUT + '.tmp', 'w') as tf:
    tf.add(OUT, arcname='omni_av_best')
os.replace(TAR_OUT + '.tmp', TAR_OUT)
TAR_SHA = sha256(TAR_OUT)
shutil.copy2(f'{OUT}/config.json', f'{H}/config.json')
if HF_OUT and not SMOKE:
    api.create_repo(HF_OUT, repo_type='dataset', private=True, exist_ok=True)
    api.upload_folder(repo_id=HF_OUT, repo_type='dataset', folder_path=OUT, commit_message=f'Omni best teacher vectors (Fushion {COMMIT[:7]})')
    print('đã upload:', HF_OUT)
note(tar=TAR_OUT, tar_sha256=TAR_SHA, hf=HF_OUT if not SMOKE else '', config_sha256=sha256(f'{OUT}/config.json'), flags=FLAGS, env=ENV, sec_total=round(time.time() - T_START, 1))
print(json.dumps({k: INV[k] for k in ('counts', 'written', 'missing_reasons', 'readback', 'tar_sha256', 'config_sha256', 'sec_encode', 'sec_total') if k in INV}, indent=1, ensure_ascii=False))
print('encode_check:', json.dumps({k: v for k, v in (INV.get('encode_check') or {}).items() if k != 'mp4_vs_clip'}, ensure_ascii=False))
print(f'''
Train UniAV (file 01 §5, user chạy; HanLi {HANLI_REF}, không sửa code):
  bộ thầy: {HF_OUT or TAR_OUT} -> giải nén / tải về datasets/omni_av_best/{{train,val}}/<id>.npz
  python train.py --output htstep_best --set "dataset.omni_emb_folder='./datasets/omni_av_best'"
''')
if SMOKE:
    print('SMOKE xong: mọi CHECK OK. Đổi SMOKE = False ở ô 0 rồi Run all.')